# HorchAI — Proof of Concept: Akustische Keystroke-Erkennung

**HorchAI v0.1** — Phase 1 (Keystroke-Erkennung) bis Phase 3 (Baseline-Klassifikation).

Dieses Notebook ist für **Google Colab** gebaut und für die Bedienung vom **Smartphone**
aus optimiert: oben rechts **"Copy to Drive"** (optional) und dann **Runtime → Run all**.
Wo eine Eingabe nötig ist (z.B. eigene Audiodatei hochladen), ist das klar markiert.

> **Sicherheits- & Forschungskontext:** Dieses Projekt untersucht akustische
> Seitenkanal-Angriffe auf Tastaturen (inspiriert von [arXiv:2504.11622](https://arxiv.org/abs/2504.11622))
> ausschließlich mit **eigenen oder ausdrücklich autorisierten** Aufnahmen. Es werden
> **keine** echten fremden Passwörter, Zugangsdaten oder nicht autorisierten Eingaben
> rekonstruiert. Details: [`docs/ethics.md`](../docs/ethics.md).

**Pipeline in diesem Notebook:**

```
Audioaufnahme → Preprocessing → Keystroke Detection → Segmentierung
  → Mel-Spektrogramme → (Phase 2) 7-Tasten-Datensatz → (Phase 3) CNN-Baseline
  → Accuracy + Confusion Matrix
```


## Setup

Installiert Abhängigkeiten und lädt die HorchAI-Module (`src/`).

In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/JimmyCore/HorchAI.git"

if IN_COLAB:
    REPO_DIR = "/content/HorchAI"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    %pip install -q -r {REPO_DIR}/requirements.txt
else:
    # Lokal ausgeführt (nicht Colab): wir gehen davon aus, dass dieses Notebook
    # bereits innerhalb des geklonten Repos liegt.
    REPO_DIR = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"HorchAI-Repo: {REPO_DIR}")
print(f"In Colab: {IN_COLAB}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

from src import audio, detection, features, dataset, model as model_mod, evaluation

SEED = 42
np.random.seed(SEED)
model_mod.set_seed(SEED)

print("Module geladen, Seed gesetzt:", SEED)


---
## Phase 1 — Minimaler PoC: Keystroke-Erkennung in einer einzelnen Aufnahme

Lade eine eigene kurze Audioaufnahme hoch (z.B. 5-10 Tastenanschläge, ein paar
Sekunden lang, `.wav`). Diese Phase zeigt die komplette Detektions-Pipeline an
**einer** Datei, bevor in Phase 2 der volle Datensatz gebaut wird.


In [ ]:
# --- EINGABE ERFORDERLICH: Audiodatei hochladen ---
if IN_COLAB:
    from google.colab import files

    print("Bitte eine .wav-Datei mit ein paar Tastenanschlägen hochladen...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("Keine Datei hochgeladen.")
    demo_path = list(uploaded.keys())[0]
else:
    demo_path = "data/raw/a/take1.wav"  # <-- bei lokaler Ausführung anpassen

print(f"Verwende Datei: {demo_path}")


In [ ]:
y, sr = audio.load_audio(demo_path, sr=16000)
y = audio.normalize(y)

print(f"Sample rate: {sr} Hz")
print(f"Dauer: {audio.get_duration(y, sr):.2f} s")
print(f"Samples: {len(y)}")


### Waveform

In [ ]:
t = np.arange(len(y)) / sr

plt.figure(figsize=(12, 3))
plt.plot(t, y, linewidth=0.5)
plt.xlabel("Zeit (s)")
plt.ylabel("Amplitude")
plt.title("Waveform")
plt.tight_layout()
plt.show()


### Mel-Spektrogramm (gesamte Aufnahme)

In [ ]:
mel_db_full = features.mel_spectrogram(y, sr, n_mels=64, n_fft=1024, hop_length=256)

plt.figure(figsize=(12, 4))
plt.imshow(
    mel_db_full,
    aspect="auto",
    origin="lower",
    cmap="magma",
    extent=[0, audio.get_duration(y, sr), 0, mel_db_full.shape[0]],
)
plt.xlabel("Zeit (s)")
plt.ylabel("Mel-Band")
plt.title("Mel-Spektrogramm")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()


### Keystroke-Erkennung

Energie-basierte Peak-Erkennung (`src/detection.py`). Falls zu viele/wenige
Anschläge erkannt werden, `threshold_factor` anpassen (kleiner = empfindlicher):
`detection.detect_keystrokes(y, sr, threshold_factor=2.0)`.


In [ ]:
keystrokes = detection.detect_keystrokes(y, sr)
print(f"{len(keystrokes)} mögliche Tastenanschläge erkannt")


In [ ]:
envelope, env_times = detection.energy_envelope(y, sr)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, y, linewidth=0.5, color="gray")
for k in keystrokes:
    axes[0].axvline(k.time_sec, color="red", linewidth=0.8, alpha=0.7)
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Waveform mit erkannten Anschlägen (rot)")

axes[1].plot(env_times, envelope, color="steelblue")
for k in keystrokes:
    axes[1].axvline(k.time_sec, color="red", linewidth=0.8, alpha=0.7)
axes[1].set_xlabel("Zeit (s)")
axes[1].set_ylabel("Energie (RMS)")
axes[1].set_title("Energie-Hüllkurve mit erkannten Peaks")

plt.tight_layout()
plt.show()


In [ ]:
for i, k in enumerate(keystrokes):
    print(f"Anschlag {i + 1:2d}: t = {k.time_sec:6.3f} s   Energie = {k.energy:.4f}")


---
## Phase 2 — Kontrollierter 7-Tasten-Datensatz

Baut einen gelabelten Datensatz aus `data/raw/<taste>/*.wav` (siehe
[`docs/recording_protocol.md`](../docs/recording_protocol.md) für das genaue
Aufnahme-Protokoll). Erwartete Struktur:

```
data/raw/
├── a/take1.wav, take2.wav, ...
├── s/...
├── d/...
├── f/...
├── j/...
├── k/...
└── l/...
```

**Zwei Wege, die Daten hierher zu bekommen (Colab, ohne Terminal):**
1. Ordner `data/raw/` einmalig in Google Drive ablegen und unten mounten (empfohlen
   für den vollen Datensatz).
2. Für einen schnellen Test: Zip-Datei mit der `data/raw/`-Struktur hochladen und
   entpacken (Zelle darunter, alternativ zu Drive).


In [ ]:
# --- EINGABE ERFORDERLICH: Pfad zu den Rohdaten festlegen ---
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    RAW_DATA_DIR = "/content/drive/MyDrive/HorchAI/data/raw"  # <-- ggf. anpassen
else:
    RAW_DATA_DIR = "data/raw"

print(f"Erwarte Rohdaten unter: {RAW_DATA_DIR}")
print("Falls der Ordner nicht existiert: docs/recording_protocol.md befolgen,")
print("Aufnahmen dort ablegen und diese Zelle erneut ausführen.")


Alternative zu Drive: eine `.zip` mit der `data/raw/`-Struktur hochladen und hier
entpacken (Zelle ausführen, dann im folgenden Upload-Dialog die Zip auswählen).


In [ ]:
# Optional: Zip-Upload statt Drive-Mount
UPLOAD_ZIP = False  # auf True setzen, um diese Zelle zu nutzen

if UPLOAD_ZIP and IN_COLAB:
    import zipfile

    from google.colab import files

    uploaded_zip = files.upload()
    zip_name = list(uploaded_zip.keys())[0]
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall("/content/data_raw")
    RAW_DATA_DIR = "/content/data_raw"
    print(f"Entpackt nach: {RAW_DATA_DIR}")


In [ ]:
KEYS = dataset.DEFAULT_KEYS  # ["a", "s", "d", "f", "j", "k", "l"]
build_config = dataset.DatasetBuildConfig(target_sr=16000)

items = dataset.build_dataset_from_raw(RAW_DATA_DIR, keys=KEYS, config=build_config)
print(f"{len(items)} Tastenanschläge aus {RAW_DATA_DIR} extrahiert")

meta_df = dataset.to_metadata_df(items)
print(meta_df.groupby("label").size().rename("Anzahl"))


In [ ]:
# Metadaten speichern (keine Audiodaten -> sicher committable, siehe data/README.md)
meta_df.to_csv("dataset_metadata.csv", index=False)
print("Metadaten gespeichert: dataset_metadata.csv")


### Reproduzierbarer Train/Validation/Test-Split (stratifiziert nach Taste)

In [ ]:
labels = [it.label for it in items]
splits = dataset.stratified_split(labels, val_size=0.15, test_size=0.15, seed=SEED)

print(f"Train: {len(splits.train_idx)}  Val: {len(splits.val_idx)}  Test: {len(splits.test_idx)}")


### Feature-Extraktion: Mel-Spektrogramme je Anschlag

In [ ]:
segments = np.stack([it.segment for it in items])
mel_specs = features.batch_mel_spectrograms(segments, sr=build_config.target_sr)
print(f"Mel-Spektrogramme: {mel_specs.shape}  (N, n_mels, n_frames)")

label_encoder = dataset.LabelEncoder(labels)
y_encoded = label_encoder.encode(labels)
print(f"Klassen: {label_encoder.classes_}")


---
## Phase 3 — CNN-Baseline

Kleines CNN (`src/model.py:SimpleCNN`) auf den Mel-Spektrogrammen. Bewusst klein
gehalten, da der Datensatz (Phase 2) ebenfalls klein ist.


In [ ]:
X = torch.tensor(mel_specs).unsqueeze(1)  # (N, 1, n_mels, n_frames)
Y = torch.tensor(y_encoded)

X_train, Y_train = X[splits.train_idx], Y[splits.train_idx]
X_val, Y_val = X[splits.val_idx], Y[splits.val_idx]
X_test, Y_test = X[splits.test_idx], Y[splits.test_idx]

print(f"Train: {tuple(X_train.shape)}  Val: {tuple(X_val.shape)}  Test: {tuple(X_test.shape)}")


In [ ]:
EPOCHS = 30
BATCH_SIZE = 8
LEARNING_RATE = 1e-3

clf = model_mod.SimpleCNN(n_classes=len(label_encoder.classes_))
optimizer = torch.optim.Adam(clf.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    clf.train()
    perm = torch.randperm(len(X_train))
    epoch_loss = 0.0
    for i in range(0, len(X_train), BATCH_SIZE):
        idx = perm[i : i + BATCH_SIZE]
        xb, yb = X_train[idx], Y_train[idx]
        optimizer.zero_grad()
        out = clf(xb)
        loss = loss_fn(out, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    epoch_loss /= max(1, len(X_train))

    clf.eval()
    with torch.no_grad():
        val_pred = clf(X_val).argmax(1)
        val_acc = (val_pred == Y_val).float().mean().item() if len(X_val) else float("nan")

    history["train_loss"].append(epoch_loss)
    history["val_acc"].append(val_acc)
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:2d}  train_loss={epoch_loss:.4f}  val_acc={val_acc:.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(history["train_loss"])
axes[0].set_title("Train Loss")
axes[0].set_xlabel("Epoch")
axes[1].plot(history["val_acc"])
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()


### Evaluation auf dem Test-Split: Accuracy, Precision/Recall, Confusion Matrix

In [ ]:
clf.eval()
with torch.no_grad():
    test_pred = clf(X_test).argmax(1).numpy()

y_true = label_encoder.decode(Y_test.numpy())
y_pred = label_encoder.decode(test_pred)

result = evaluation.evaluate(y_true, y_pred, label_encoder.classes_)
evaluation.print_report(result)


In [ ]:
evaluation.plot_confusion_matrix(result)
plt.show()


---
## Experiment dokumentieren

Trag das Ergebnis dieses Laufs in `experiments/` ein (siehe
[`experiments/template.md`](../experiments/template.md)) — Ziel, Dataset-Version,
Parameter, Modell, Git-Commit, Resultate, Interpretation, nächster Schritt. So bleibt
jeder Lauf nachvollziehbar und vergleichbar.

## Nächste Schritte

Mit einer funktionierenden, reproduzierbaren Baseline (dieses Notebook) ist
**HorchAI v0.1** erreicht. Mögliche nächste Schritte (noch nicht in diesem Notebook):

- **Phase 4** — Vision Transformer auf denselben Mel-Spektrogrammen, fairer Vergleich
  mit identischem Split.
- **Phase 5** — Noise-Robustness: synthetisches Rauschen in kontrollierten Stufen
  hinzufügen, Accuracy-Abfall messen und visualisieren.
- **Phase 6** — separater, klar getrennter Forschungszweig zu Sprachmodell-Korrektur
  auf synthetischen Testsätzen (keine echten Zugangsdaten).

Bevor eine dieser Phasen begonnen wird: diese Baseline mit **echten** Aufnahmen
(nicht nur synthetischen Testdaten) laufen lassen und die Ergebnisse in
`experiments/` festhalten.
